In [1]:
library(ggplot2)
library(dplyr)
# Load the required libraries
library(ggpubr)
# load dplyr and tidyr library

library(tidyr)
library(parallel)
library(future)
library(furrr)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [2]:
setwd("/mnt/home3/miska/nm667/scratch/inProgress/mice_PiRNA/analysis/sRNA_deseq/sRNA_STAR_testing")

In [3]:
data <- read.csv("/mnt/home3/miska/nm667/scratch/inProgress/mice_PiRNA/results/TEtranscriptCount/all_featureCounts_TE.tab", header=FALSE,sep='\t')
data

V1,V2,V3,V4,V5,V6,V7,V8
<chr>,<chr>,<int>,<int>,<chr>,<int>,<dbl>,<chr>
pacBio/129S1_SvImJ/129S1_SvImJ-12.5dpp.1.featureCounts,,NA,NA,,NA,NA,
B3A SINE/B2 129S1_SvImJ#1#100 995-1168,129S1_SvImJ#1#100,995,1168,-,174,0,pacBio/129S1_SvImJ/129S1_SvImJ-12.5dpp.1.featureCounts
RSINE1 SINE/B4 129S1_SvImJ#1#100 1451-1605,129S1_SvImJ#1#100,1451,1605,-,155,0,pacBio/129S1_SvImJ/129S1_SvImJ-12.5dpp.1.featureCounts
(AAAG)n Simple_repeat 129S1_SvImJ#1#100 1883-2008,129S1_SvImJ#1#100,1883,2008,+,126,0,pacBio/129S1_SvImJ/129S1_SvImJ-12.5dpp.1.featureCounts
(CT)n Simple_repeat 129S1_SvImJ#1#100 2394-2436,129S1_SvImJ#1#100,2394,2436,+,43,0,pacBio/129S1_SvImJ/129S1_SvImJ-12.5dpp.1.featureCounts
(AGAA)n Simple_repeat 129S1_SvImJ#1#100 2463-2530,129S1_SvImJ#1#100,2463,2530,+,68,0,pacBio/129S1_SvImJ/129S1_SvImJ-12.5dpp.1.featureCounts
B3 SINE/B2 129S1_SvImJ#1#100 2769-2970,129S1_SvImJ#1#100,2769,2970,+,202,0,pacBio/129S1_SvImJ/129S1_SvImJ-12.5dpp.1.featureCounts
(GT)n Simple_repeat 129S1_SvImJ#1#100 3445-3544,129S1_SvImJ#1#100,3445,3544,+,100,0,pacBio/129S1_SvImJ/129S1_SvImJ-12.5dpp.1.featureCounts
(ATTTAT)n Simple_repeat 129S1_SvImJ#1#100 3830-3845,129S1_SvImJ#1#100,3830,3845,+,16,0,pacBio/129S1_SvImJ/129S1_SvImJ-12.5dpp.1.featureCounts


In [4]:
split_name <- function(row) {
    separate(row, V1, c('V9', 'V10', 'V11', 'V12'), sep = " ")
}

# Determine the number of cores to use
num_cores <- 1

# Split the data into chunks for each core
chunks <- split(data, cut(1:nrow(data), num_cores, labels = FALSE))

# Process in parallel
results <- mclapply(chunks, function(chunk) split_name(chunk), mc.cores = num_cores)

# Combine results back together
data <- bind_rows(results)


ERROR: Error in cut.default(1:nrow(data), num_cores, labels = FALSE): invalid number of intervals


In [ ]:
data 

<0 x 0 matrix>

In [ ]:
# Use multicore processing if on Unix/Linux, or multiprocess (which is sequential) if on Windows
plan(multicore)
# Split the data into chunks (for this example, let's say 4 chunks)
chunks <- split(data, cut(1:nrow(data), 4, labels = FALSE))

# Process each chunk in parallel
results <- future_map_dfr(chunks, ~ .x %>%
    group_by(V8, V9, V10) %>%
    summarise(summed_V7 = sum(V7, na.rm = TRUE)) %>%
    ungroup())


ERROR: [1m[33mError[39m:[22m
[1m[22m[36mℹ[39m In index: 1.
[36mℹ[39m With name: 1.
[1mCaused by error in `group_by()`:[22m
[1m[22m[33m![39m Must group by variables found in `.data`.
Column `V8` is not found.
Column `V9` is not found.
Column `V10` is not found.


In [ ]:
new_data <- results %>%
  group_by(V8, V9, V10) %>%
  summarise(summed_V7 = sum(summed_V7, na.rm = TRUE)) %>%
  ungroup()
new_data

In [ ]:


write.csv(new_data, "TE_samples_repeat.csv", row.names=FALSE)

ERROR: Error in eval(expr, p): object 'new_data' not found
